# Chương 3 — Phân tích khám phá dữ liệu (EDA)

**Thành viên 1 — Data Engineer**  
**Dữ liệu:** VCB, FPT, HPG, VIC, VNM — 2015/2016 → 2026-04 (nguồn: vnstock/VCI)

## Nội dung
1. Tổng quan dữ liệu (shape, date range, missing values)
2. Biểu đồ giá đóng cửa theo thời gian
3. Phân tích lợi suất hàng ngày (daily returns)
4. Ma trận tương quan giữa các mã
5. Kiểm định tính dừng — ADF Test
6. Phân rã chuỗi thời gian (Trend / Seasonal / Residual)
7. Trực quan hoá chỉ số kỹ thuật
8. Phân chia Train / Test
9. Bảng tóm tắt cho báo cáo

---
## 0 — Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#2563EB', '#DC2626', '#16A34A', '#D97706', '#7C3AED']

In [ ]:
ROOT         = Path('..').resolve()
RAW_DIR      = ROOT / 'data' / 'raw'
FEATURED_DIR = ROOT / 'data' / 'processed' / 'featured'
SPLITS_DIR   = ROOT / 'data' / 'processed' / 'splits'
PLOTS_DIR    = ROOT / 'results' / 'eda' / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ['VCB', 'FPT', 'HPG', 'VIC', 'VNM']
COLORS  = dict(zip(TICKERS, PALETTE))

# Load tất cả dữ liệu raw và featured
raw      = {t: pd.read_csv(RAW_DIR / f'{t}.csv', parse_dates=['date']) for t in TICKERS}
featured = {t: pd.read_csv(FEATURED_DIR / f'{t}_featured.csv', parse_dates=['date']) for t in TICKERS}
split_info = json.loads((SPLITS_DIR / 'split_info.json').read_text())

print('Dữ liệu đã load xong.')

---
## 1 — Tổng quan dữ liệu

In [ ]:
rows = []
for t in TICKERS:
    df = raw[t].sort_values('date')
    rows.append({
        'Mã':           t,
        'Từ ngày':      str(df['date'].min().date()),
        'Đến ngày':     str(df['date'].max().date()),
        'Số phiên':     len(df),
        'Thiếu (close)': int(df['close'].isna().sum()),
        'Giá min':      round(df['close'].min(), 2),
        'Giá max':      round(df['close'].max(), 2),
        'Giá TB':       round(df['close'].mean(), 2),
    })

overview_df = pd.DataFrame(rows)
print(overview_df.to_string(index=False))

In [ ]:
# Thống kê mô tả cho close price
desc = pd.DataFrame({t: raw[t]['close'].describe() for t in TICKERS}).round(2)
print('Thống kê mô tả — Giá đóng cửa:')
print(desc.to_string())

---
## 2 — Biểu đồ giá đóng cửa theo thời gian

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=False)

for ax, ticker in zip(axes, TICKERS):
    df = raw[ticker].sort_values('date')
    ax.plot(df['date'], df['close'], color=COLORS[ticker], lw=1.0)
    ax.set_title(ticker, fontsize=12, fontweight='bold', loc='left')
    ax.set_ylabel('Giá (VND × nghìn)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.grid(axis='y', ls=':', alpha=0.4)

axes[-1].set_xlabel('Năm')
fig.suptitle('Giá đóng cửa 5 mã chứng khoán VN (2015–2026)', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / '01_close_price_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Giá chuẩn hoá (base = 100) để so sánh hiệu suất tương đối
fig, ax = plt.subplots(figsize=(14, 5))

for ticker in TICKERS:
    df = raw[ticker].sort_values('date').dropna(subset=['close'])
    normalized = df['close'] / df['close'].iloc[0] * 100
    ax.plot(df['date'], normalized, label=ticker, color=COLORS[ticker], lw=1.3)

ax.axhline(100, color='gray', ls='--', lw=0.8, alpha=0.6)
ax.set_title('Hiệu suất tích luỹ (base = 100 tại ngày đầu tiên)', pad=10)
ax.set_ylabel('Chỉ số (base = 100)')
ax.set_xlabel('Năm')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.grid(axis='y', ls=':', alpha=0.4)
plt.tight_layout()
fig.savefig(PLOTS_DIR / '02_normalized_price.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 — Phân tích lợi suất hàng ngày (Daily Returns)

In [ ]:
returns = {}
for t in TICKERS:
    df = raw[t].sort_values('date').dropna(subset=['close'])
    returns[t] = df.set_index('date')['close'].pct_change().dropna() * 100  # đơn vị %

# Thống kê lợi suất
ret_stats = pd.DataFrame({
    t: {
        'Mean (%)':     round(returns[t].mean(), 4),
        'Std (%)':      round(returns[t].std(), 4),
        'Skewness':     round(returns[t].skew(), 4),
        'Kurtosis':     round(returns[t].kurtosis(), 4),
        'Min (%)':      round(returns[t].min(), 4),
        'Max (%)':      round(returns[t].max(), 4),
    } for t in TICKERS
})
print('Thống kê lợi suất hàng ngày (%):')
print(ret_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)

for ax, ticker in zip(axes, TICKERS):
    r = returns[ticker]
    ax.hist(r, bins=60, color=COLORS[ticker], alpha=0.7, edgecolor='white', lw=0.3)
    ax.axvline(r.mean(), color='black', ls='--', lw=1.2, label=f'Mean={r.mean():.3f}%')
    ax.set_title(ticker, fontweight='bold')
    ax.set_xlabel('Return (%)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Tần số')
fig.suptitle('Phân phối lợi suất hàng ngày', fontsize=13)
plt.tight_layout()
fig.savefig(PLOTS_DIR / '03_returns_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Volatility rolling 30 ngày
fig, ax = plt.subplots(figsize=(14, 4))

for ticker in TICKERS:
    roll_vol = returns[ticker].rolling(30).std()
    ax.plot(roll_vol.index, roll_vol, label=ticker, color=COLORS[ticker], lw=1.0, alpha=0.85)

ax.set_title('Biến động lăn 30 ngày (Rolling 30-day Volatility)', pad=10)
ax.set_ylabel('Độ lệch chuẩn lợi suất (%)')
ax.set_xlabel('Năm')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.grid(axis='y', ls=':', alpha=0.4)
plt.tight_layout()
fig.savefig(PLOTS_DIR / '04_rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 — Ma trận tương quan

In [ ]:
# Tương quan giá đóng cửa
price_df  = pd.DataFrame({t: raw[t].set_index('date')['close'] for t in TICKERS}).dropna()
return_df = price_df.pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, title in [
    (axes[0], price_df.corr(),  'Tương quan giá đóng cửa'),
    (axes[1], return_df.corr(), 'Tương quan lợi suất hàng ngày'),
]:
    sns.heatmap(
        df, annot=True, fmt='.2f', cmap='RdYlGn',
        vmin=-1, vmax=1, center=0,
        square=True, linewidths=0.5,
        ax=ax, cbar_kws={'shrink': 0.8},
    )
    ax.set_title(title, pad=10)

plt.tight_layout()
fig.savefig(PLOTS_DIR / '05_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 — Kiểm định tính dừng (ADF Test)

**Augmented Dickey-Fuller Test**  
- H₀: chuỗi có nghiệm đơn vị (không dừng)  
- H₁: chuỗi dừng  
- Bác bỏ H₀ khi **p-value < 0.05**

In [ ]:
def run_adf(series: pd.Series, name: str) -> dict:
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Chuỗi':       name,
        'ADF Stat':    round(result[0], 4),
        'p-value':     round(result[1], 6),
        'Lags':        result[2],
        'Critical 1%': round(result[4]['1%'], 4),
        'Critical 5%': round(result[4]['5%'], 4),
        'Dừng?':       '✅ Có' if result[1] < 0.05 else '❌ Không',
    }

adf_rows = []
for ticker in TICKERS:
    series = raw[ticker].sort_values('date').set_index('date')['close']
    adf_rows.append(run_adf(series,                   f'{ticker} (giá)'))
    adf_rows.append(run_adf(series.pct_change() * 100, f'{ticker} (lợi suất)'))

adf_df = pd.DataFrame(adf_rows)
print(adf_df.to_string(index=False))

**Nhận xét:** Chuỗi giá thường không dừng (p > 0.05), chuỗi lợi suất dừng (p < 0.05). Đây là đặc tính điển hình của dữ liệu chứng khoán — cần lưu ý khi áp dụng ARIMA.

---
## 6 — Phân rã chuỗi thời gian (Seasonal Decomposition)

In [ ]:
# Phân rã cho từng mã — period=252 (xấp xỉ số phiên giao dịch/năm)
PERIOD = 252

for ticker in TICKERS:
    series = (
        raw[ticker].sort_values('date')
        .set_index('date')['close']
        .dropna()
        .asfreq('B')           # Business day frequency
        .ffill()               # Fill gaps từ holidays
    )

    decomp = seasonal_decompose(series, model='multiplicative', period=PERIOD, extrapolate_trend='freq')

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    components = [
        (series,            'Observed (Giá gốc)', COLORS[ticker]),
        (decomp.trend,      'Trend (Xu hướng)',   '#374151'),
        (decomp.seasonal,   'Seasonal (Mùa vụ)',  '#059669'),
        (decomp.resid,      'Residual (Nhiễu)',   '#9CA3AF'),
    ]
    for ax, (data, label, color) in zip(axes, components):
        ax.plot(data.index, data.values, color=color, lw=1.0)
        ax.set_ylabel(label, fontsize=9)
        ax.grid(axis='y', ls=':', alpha=0.4)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].set_xlabel('Năm')
    fig.suptitle(f'Seasonal Decomposition — {ticker} (multiplicative, period=252)', fontsize=12)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f'06_decomposition_{ticker}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 7 — Trực quan hoá chỉ số kỹ thuật

Minh hoạ trên mã **VCB** — 2 năm gần nhất để dễ đọc.

In [ ]:
DEMO_TICKER = 'VCB'
df_feat = featured[DEMO_TICKER].sort_values('date')
df_2y   = df_feat[df_feat['date'] >= '2024-01-01'].copy()

fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(4, 1, figure=fig, hspace=0.05, height_ratios=[3, 1, 1, 1])

# ── Panel 1: Giá + MA + Bollinger Bands ──
ax1 = fig.add_subplot(gs[0])
ax1.fill_between(df_2y['date'], df_2y['bb_lower'], df_2y['bb_upper'],
                 alpha=0.12, color='gray', label='Bollinger Bands')
ax1.plot(df_2y['date'], df_2y['close'],     color='#1F2937', lw=1.2, label='Close')
ax1.plot(df_2y['date'], df_2y['ma_5'],      color='#F59E0B', lw=1.0, ls='--', label='MA 5')
ax1.plot(df_2y['date'], df_2y['ma_20'],     color='#3B82F6', lw=1.0, ls='--', label='MA 20')
ax1.plot(df_2y['date'], df_2y['ma_50'],     color='#EF4444', lw=1.0, ls='--', label='MA 50')
ax1.plot(df_2y['date'], df_2y['bb_upper'],  color='gray',    lw=0.7, alpha=0.6)
ax1.plot(df_2y['date'], df_2y['bb_lower'],  color='gray',    lw=0.7, alpha=0.6)
ax1.set_ylabel('Giá (VND × nghìn)')
ax1.legend(loc='upper left', fontsize=9, ncol=3)
ax1.set_title(f'Chỉ số kỹ thuật — {DEMO_TICKER} (2024 → nay)', fontsize=12)
ax1.set_xticklabels([])

# ── Panel 2: Volume ──
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.bar(df_2y['date'], df_2y['volume'] / 1e6, color='#6B7280', alpha=0.6, width=1)
ax2.set_ylabel('Volume\n(triệu CP)', fontsize=9)
ax2.set_xticklabels([])

# ── Panel 3: RSI ──
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(df_2y['date'], df_2y['rsi_14'], color='#7C3AED', lw=1.2)
ax3.axhline(70, color='#DC2626', ls='--', lw=0.8, alpha=0.7)
ax3.axhline(30, color='#16A34A', ls='--', lw=0.8, alpha=0.7)
ax3.fill_between(df_2y['date'], 30, 70, alpha=0.05, color='gray')
ax3.set_ylim(0, 100)
ax3.set_ylabel('RSI (14)', fontsize=9)
ax3.set_xticklabels([])

# ── Panel 4: MACD ──
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.bar(df_2y['date'], df_2y['macd_hist'],
        color=np.where(df_2y['macd_hist'] >= 0, '#16A34A', '#DC2626'),
        alpha=0.7, width=1)
ax4.plot(df_2y['date'], df_2y['macd'],        color='#2563EB', lw=1.0, label='MACD')
ax4.plot(df_2y['date'], df_2y['macd_signal'], color='#F97316', lw=1.0, ls='--', label='Signal')
ax4.axhline(0, color='gray', lw=0.6)
ax4.set_ylabel('MACD', fontsize=9)
ax4.legend(loc='upper left', fontsize=8)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.savefig(PLOTS_DIR / f'07_technical_indicators_{DEMO_TICKER}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8 — Phân chia Train / Test

In [ ]:
# Lấy ngày cắt từ split_info
cut_dates = {}
for entry in split_info:
    t = entry['ticker']
    cut_dates[t] = {
        '70_30': pd.Timestamp(entry['splits']['70_30']['test_start']),
        '80_20': pd.Timestamp(entry['splits']['80_20']['test_start']),
    }

fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=False)

for ax, ticker in zip(axes, TICKERS):
    df   = raw[ticker].sort_values('date')
    c730 = cut_dates[ticker]['70_30']
    c820 = cut_dates[ticker]['80_20']

    # Vùng màu: train 70/30
    ax.axvspan(df['date'].min(), c730, alpha=0.08, color='#2563EB', label='Train 70%')
    ax.axvspan(c730, df['date'].max(), alpha=0.08, color='#DC2626', label='Test 30%')

    ax.plot(df['date'], df['close'], color='#374151', lw=1.0)
    ax.axvline(c730, color='#2563EB', ls='--', lw=1.3, label=f'70/30 cut: {c730.date()}')
    ax.axvline(c820, color='#D97706', ls='--', lw=1.3, label=f'80/20 cut: {c820.date()}')

    ax.set_title(ticker, fontsize=11, fontweight='bold', loc='left')
    ax.set_ylabel('Giá')
    ax.legend(loc='upper left', fontsize=8, ncol=2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.grid(axis='y', ls=':', alpha=0.3)

axes[-1].set_xlabel('Năm')
fig.suptitle('Phân chia Train/Test theo thời gian (không shuffle)', fontsize=13, y=1.005)
plt.tight_layout()
fig.savefig(PLOTS_DIR / '08_train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9 — Bảng tóm tắt cho Chương 3

In [ ]:
summary_rows = []
for ticker in TICKERS:
    df  = raw[ticker].sort_values('date').dropna(subset=['close'])
    ret = df['close'].pct_change().dropna() * 100
    adf_price  = adfuller(df['close'], autolag='AIC')
    adf_return = adfuller(ret, autolag='AIC')

    si = next(e for e in split_info if e['ticker'] == ticker)
    summary_rows.append({
        'Mã':               ticker,
        'Số phiên':         len(df),
        'Từ':               str(df['date'].min().date()),
        'Đến':              str(df['date'].max().date()),
        'Giá TB':           round(df['close'].mean(), 1),
        'Std lợi suất (%)': round(ret.std(), 3),
        'ADF giá (p)':      round(adf_price[1], 4),
        'ADF lợi suất (p)': round(adf_return[1], 6),
        'Train 70/30':      si['splits']['70_30']['train_rows'],
        'Test 70/30':       si['splits']['70_30']['test_rows'],
        'Train 80/20':      si['splits']['80_20']['train_rows'],
        'Test 80/20':       si['splits']['80_20']['test_rows'],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

out_csv = ROOT / 'results' / 'eda' / 'chapter3_summary.csv'
summary_df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'\nĐã lưu → {out_csv.relative_to(ROOT)}')

In [ ]:
print('Các biểu đồ đã lưu tại results/eda/plots/:')
for p in sorted(PLOTS_DIR.iterdir()):
    print(f'  {p.name}')

---
## Ghi chú viết Chương 3

### 3.1 Nguồn dữ liệu
- Nguồn chính: **vnstock** (VCI) — dữ liệu OHLCV sàn HOSE/HNX
- Fallback: **yfinance** (suffix `.VN`)
- Khoảng thời gian: ~10 năm (2015/2016 → 4/2026), ~2 648 phiên mỗi mã

### 3.2 Làm sạch dữ liệu
- Loại bỏ duplicate ngày, sắp xếp chronological
- Xóa hàng có `close = NaN` (lỗi cấu trúc)
- Forward-fill ≤3 phiên liên tiếp cho OHLCV (bù khoảng trống cuối tuần dài)

### 3.3 Feature Engineering
| Indicator | Tham số | Ý nghĩa |
|---|---|---|
| MA | 5, 20, 50 ngày | Xu hướng ngắn/trung/dài hạn |
| RSI | 14 ngày | Momentum, vùng mua/bán quá mức (30/70) |
| MACD | 12/26/9 | Sự hội tụ/phân kỳ trung bình động |
| Bollinger Bands | 20 ngày, ±2σ | Biên biến động |

### 3.4 Tính dừng
- Chuỗi giá **không dừng** (ADF p > 0.05) — đặc tính random walk
- Chuỗi lợi suất **dừng** (ADF p ≈ 0) — phù hợp mô hình thống kê

### 3.5 Chia tập dữ liệu
- Chia **theo thời gian** (không shuffle) để tránh data leakage
- Hai tỷ lệ: **70/30** (test từ 2023-02) và **80/20** (test từ 2024-03)